# EAT model (based on Wang's paper)

## Data loading, and preparing for EAT

In [1]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torchaudio
import soundfile as sf
import os
import glob
import pandas as pd
from tqdm import tqdm

class DCASEWangDataset(Dataset):
    def __init__(self, root_dir, machine_type, target_length=1024, is_train=True):
        self.root_dir = root_dir
        self.machine_type = machine_type
        self.target_length = target_length
        self.is_train = is_train
        
        # 1. Load the attributes CSV
        csv_path = os.path.join(root_dir, machine_type, "attributes_00.csv")
        self.df = pd.read_csv(csv_path)
        
        # Logic for Domain vs Attribute
        val_cols = [c for c in self.df.columns if c.endswith('v')]
        if len(val_cols) > 0:
            self.df['label_str'] = self.df[val_cols].astype(str).agg('_'.join, axis=1)
            self.task_type = "attribute"
        else:
            if 'domain' in self.df.columns:
                self.df['label_str'] = self.df['domain'].astype(str)
            else:
                self.df['label_str'] = self.df['file_name'].apply(
                    lambda x: "source" if "source" in x else "target"
                )
            self.task_type = "domain"

        unique_labels = sorted(self.df['label_str'].unique())
        self.label_to_id = {label: i for i, label in enumerate(unique_labels)}
        self.num_classes = len(self.label_to_id)
        self.label_lookup = {
            os.path.basename(row['file_name']): self.label_to_id[row['label_str']] 
            for _, row in self.df.iterrows()
        }
        
        subset = "train" if is_train else "test"
        search_path = os.path.join(root_dir, machine_type, subset, "*.wav")
        self.file_list = glob.glob(search_path)

    def __len__(self):
        return len(self.file_list)

    def preprocess_audio(self, wav, sr=16000):
        """Converts raw waveform to normalized Mel-spectrogram."""
        # Ensure 2D for kaldi.fbank (1, samples)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
            
        # 1. Convert to mel-spectrogram
        mel = torchaudio.compliance.kaldi.fbank(
            wav, htk_compat=True, sample_frequency=sr, use_energy=False,
            window_type='hanning', num_mel_bins=128, dither=0.0, frame_shift=10
        ) # Output shape: [T, 128]
        
        # 2. Add channel dimension and pad/truncate
        mel = mel.unsqueeze(0) # [1, T, 128]
        n_frames = mel.shape[1]
        if n_frames < self.target_length:
            mel = torch.nn.functional.pad(mel, (0, 0, 0, self.target_length - n_frames))
        else:
            mel = mel[:, :self.target_length, :]
        
        # 3. EAT Global Normalization
        mel = (mel - (-4.268)) / (4.569 * 2)
        return mel

    def __getitem__(self, idx):
        wav_path = self.file_list[idx]
        fname = os.path.basename(wav_path)
        domain_id = 0 if "source" in fname else 1
        label_id = self.label_lookup.get(fname, 0)
        
        wav, sr = sf.read(wav_path)
        wav = torch.tensor(wav).float()
        
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        
        wav = wav - wav.mean()
        
        # Preprocess here so DataLoader returns the final model input
        mel = self.preprocess_audio(wav)
        
        return mel, torch.tensor(label_id), torch.tensor(domain_id)

In [2]:
# 1. Initialize for Bearing (or any machine type)
# Make sure the path points to your actual data folder
dataset = DCASEWangDataset("data/dcase2023t2/dev_data/raw", "bearing", is_train=True)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# 2. Check the metadata extraction
print(f"--- Dataset Info ---")
print(f"Machine Type: {dataset.machine_type}")
print(f"Task Detected: {dataset.task_type}")  # Should say 'attribute' for bearing
print(f"Total classes (N): {dataset.num_classes}")
print(f"Mapping (First 3): {list(dataset.label_to_id.items())[:3]}")

# 3. Test a single batch
wav, label_id, domain_id = next(iter(dataloader))

print(f"\n--- Batch Check ---")
print(f"Waveform shape: {wav.shape}")        # Expected: [batch, samples]
print(f"Labels (Targets): {label_id}")       # The IDs for ArcFace
print(f"Domains (0=Src, 1=Trg): {domain_id}") # The domain info for AUC calculation

# 4. Verify specific domain counts in the whole dataset
source_count = sum(1 for f in dataset.file_list if "source" in f)
target_count = sum(1 for f in dataset.file_list if "target" in f)
print(f"\n--- File Distribution ---")
print(f"Source files: {source_count}")
print(f"Target files: {target_count}")

--- Dataset Info ---
Machine Type: bearing
Task Detected: attribute
Total classes (N): 29
Mapping (First 3): [('11_A', 0), ('13_A', 1), ('15_A', 2)]

--- Batch Check ---
Waveform shape: torch.Size([4, 1, 1024, 128])
Labels (Targets): tensor([ 1,  1,  5, 18])
Domains (0=Src, 1=Trg): tensor([0, 0, 0, 0])

--- File Distribution ---
Source files: 990
Target files: 10


In [3]:
import torch.nn as nn
import torch.nn.functional as F
import math

class ArcFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcFaceHead, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s # Scaling factor
        self.m = m # Margin
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        # 1. Norm the weights and input
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # 2. Add margin
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * math.cos(self.m) - sine * math.sin(self.m)
        
        # 3. Only apply margin to the 'correct' class
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        
        return output * self.s

In [4]:
import torch
import torch.nn as nn

class EAT_FFT_Model(nn.Module):
    def __init__(self, backbone, num_classes, embed_dim=768):
        super(EAT_FFT_Model, self).__init__()
        self.backbone = backbone  # This should be the EAT Encoder
        self.classifier = ArcFaceHead(in_features=embed_dim, out_features=num_classes)
        
    def forward(self, x, labels=None):
        # x shape: [Batch, 1, Time, Mel]
        # EAT outputs patch embeddings [Batch, Num_Patches, Embed_Dim]
        patch_embeddings = self.backbone(x) 
        
        # Average pooling of all patch embeddings [cite: 45]
        embedding = torch.mean(patch_embeddings, dim=1) 
        
        if labels is not None:
            # During training: return ArcFace logits
            return self.classifier(embedding, labels)
        
        # During testing: return the embedding for KNN [cite: 13]
        return embedding

In [5]:
from transformers import AutoModel

model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = AutoModel.from_pretrained(model_id, trust_remote_code=True)

model = EAT_FFT_Model(backbone, num_classes=dataset.num_classes).to(device)

model.train()

EAT_FFT_Model(
  (backbone): EATModel(
    (model): EAT(
      (local_encoder): PatchEmbed_new(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (pos_drop): Dropout(p=0.0, inplace=True)
      (fixed_positional_encoder): FixedPositionalEncoder()
      (blocks): ModuleList(
        (0-11): 12 x AltBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): AltAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (act): GELU(approximate='none')
            (drop1): Dropout(p=0.0, inplac